# 🧠 CalRetail — Inventory Health Monitoring
## Goal
Assign SKU status indicators (Healthy, At Risk, Critical) from a genuine composite of stock
coverage, overstock, and real supplier reliability.

## Algorithmic Explanation
**Multi-Metric Composite Inventory Scoring**
1. Calculate daily purchase consumption velocity rate from real transactions.
2. Compute days cover (current stock / daily velocity) and map to stockout risk via an inverse
   sigmoid.
3. Blend stockout risk, overstock flag, and the product's real supplier reliability score using
   PCA-derived weights (`adaptive_thresholds.get_inventory_health_weights`) into one composite
   health score — previously the "composite" score used only stockout risk and never touched
   overstock or supplier reliability at all, despite loading the suppliers table.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from backend.utils.adaptive_thresholds import get_inventory_health_weights

inv = load_table('inventory')
tx = load_table('transactions')
suppliers = load_table('suppliers')
prod = load_table('products')

# Compute daily velocity over 30 days
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])
max_date = tx['transaction_date'].max()
recent_tx = tx[tx['transaction_date'] >= (max_date - pd.Timedelta(days=30))]

velocity = recent_tx.groupby('product_id')['quantity'].sum().reset_index()
velocity['daily_velocity'] = velocity['quantity'] / 30.0

# Real supplier reliability per product (previously loaded but never used).
product_supplier_map = dict(zip(prod['product_id'], prod['supplier_id']))
supplier_reliability_map = dict(zip(suppliers['supplier_id'], suppliers['reliability_score']))
DEFAULT_RELIABILITY = float(suppliers['reliability_score'].median())

# PCA-derived composite weights for [stockout_risk, overstock_flag, supplier
# reliability] — replaces a health score that only ever looked at stockout risk.
W_STOCKOUT, W_OVERSTOCK, W_RELIABILITY = get_inventory_health_weights()

print(f"Loaded stock information. Daily velocity calculated for {len(velocity)} products.")
print(f"Composite health weights (data-derived): stockout={W_STOCKOUT:.2f}, overstock={W_OVERSTOCK:.2f}, reliability={W_RELIABILITY:.2f}")

In [ ]:
def compute_inventory_health():
    # Merge stock details with velocity
    health_df = pd.merge(inv, velocity, on='product_id', how='left')
    health_df['daily_velocity'] = health_df['daily_velocity'].fillna(0.1) # default min
    
    # Calculate days cover
    health_df['days_cover'] = health_df['stock_qty'] / health_df['daily_velocity']
    
    # Vectorised, not row-by-row. This runs over every stock position in the
    # estate — 25,000 rows — and an iterrows() loop building a dict per row took
    # ~27s on a small host, which is most of a page load spent on arithmetic
    # pandas does in one pass.
    cover = health_df['days_cover']
    rop = health_df['reorder_point']

    # stockout risk function (sigmoid of difference)
    stockout_risk = 1.0 / (1.0 + np.exp((cover - rop) * 0.2))

    # Calculate overstock
    max_stk = health_df['max_stock'].fillna(9999.0)
    overstock_flag = (health_df['stock_qty'] > max_stk).astype(int)

    # Real supplier reliability for this SKU's actual supplier
    reliability = (health_df['product_id']
                   .map(product_supplier_map)
                   .map(supplier_reliability_map)
                   .fillna(DEFAULT_RELIABILITY))

    # Genuine composite score: PCA-derived weights blending stockout risk,
    # overstock, and real supplier reliability (not stockout risk alone).
    health_score = np.clip(
        W_STOCKOUT * (1.0 - stockout_risk) +
        W_OVERSTOCK * (1.0 - overstock_flag) +
        W_RELIABILITY * reliability,
        0.0, 1.0
    )

    label = np.where(health_score < 0.4, "Critical",
                     np.where(health_score < 0.7, "At Risk", "Healthy"))

    out = pd.DataFrame({
        "product_id": health_df['product_id'],
        "store_id": health_df['store_id'].fillna("").astype(str),
        "warehouse_id": health_df['warehouse_id'].fillna("").astype(str),
        "location_type": health_df['location_type'].fillna("").astype(str),
        "stock_level": health_df['stock_qty'].astype(int),
        "days_cover": cover.astype(float).round(1),
        "stockout_risk": stockout_risk.astype(float).round(3),
        "supplier_reliability": reliability.astype(float).round(3),
        "health_score": health_score.astype(float).round(2),
        "risk_label": label,
        "overstock_flag": overstock_flag.astype(int),
    })
    results = out.to_dict(orient='records')
    return results

inventory_health = compute_inventory_health()
print("Inventory Health metric:", json.dumps(inventory_health[0], indent=2))

In [ ]:
print("=== CALRETAIL INVENTORY CONTROL RADAR ===")
health_summary_df = pd.DataFrame(inventory_health)
# Group stats
print("SKU Classification Volume:")
print(health_summary_df['risk_label'].value_counts())
print("\nCritical Stock Items needing Replenishment:")
critical_items = health_summary_df[health_summary_df['risk_label'] == 'Critical'].sort_values('days_cover')
print(critical_items[['product_id', 'stock_level', 'days_cover', 'stockout_risk']].head(5).to_string(index=False))
